In [62]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [60]:
# STEP 1: Prepare the tools
rng = np.random.default_rng(42)  

In [28]:
# STEP 2: Define the catalogue
base_prices = {
    "Laptop": 900,
    "Phone": 600,
    "Tablet": 350,
    "Headphones": 120,
    "Monitor": 280,
}
products = list(base_prices.keys())
product_weights = [0.20, 0.28, 0.18, 0.17, 0.17]
regions = ["North", "South", "East", "West"]

In [30]:
# STEP 3: Generate 5,000 sales rows
n_rows = 5000
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-12-31")
date_range = pd.date_range(start_date, end_date, freq="D")

sales_dates = rng.choice(date_range, size=n_rows)
sales_products = rng.choice(products, size=n_rows, p=product_weights)
sales_regions = rng.choice(regions, size=n_rows)
units_sold = rng.integers(1, 6, size=n_rows)  # Random integer from 1 to 5

df = pd.DataFrame({
    "OrderID": [f"ORD{i:05d}" for i in range(1, n_rows + 1)],
    "Date": sales_dates,
    "Product": sales_products,
    "Region": sales_regions,
    "UnitsSold": units_sold,
})

In [32]:
# STEP 4: Price each row (base price +/- approximately 5%)
df["BasePrice"] = df["Product"].map(base_prices)
price_factor = rng.uniform(0.95, 1.05, size=n_rows)
df["UnitPrice"] = (df["BasePrice"] * price_factor).round(2)

In [34]:
# STEP 5: Add a holiday boost
# About half of November and December rows receive one extra unit.
holiday_mask = df["Date"].dt.month.isin([11, 12])
holiday_boost_mask = holiday_mask & (rng.random(n_rows) < 0.5)
df.loc[holiday_boost_mask, "UnitsSold"] += 1

In [36]:
# STEP 6: Calculate revenue, sort by date, and save sales.csv
df["Revenue"] = (df["UnitsSold"] * df["UnitPrice"]).round(2)
df = df.sort_values(["Date", "OrderID"]).reset_index(drop=True)
df.to_csv("sales.csv", index=False)

In [38]:
# STEP 7: Summarise with Pandas
monthly_revenue = (
    df.set_index("Date")["Revenue"]
      .resample("MS")
      .sum()
      .rename("Revenue")
      .to_frame()
)
monthly_revenue["Rolling3M"] = (
    monthly_revenue["Revenue"].rolling(window=3, min_periods=1).mean()
)

product_revenue = (
    df.groupby("Product")["Revenue"]
      .sum()
      .sort_values(ascending=False)
)
region_revenue = (
    df.groupby("Region")["Revenue"]
      .sum()
      .sort_values(ascending=False)
)
product_region = pd.pivot_table(
    df,
    values="Revenue",
    index="Product",
    columns="Region",
    aggfunc="sum",
    fill_value=0,
)

monthly_revenue.to_csv("monthly_revenue.csv")
product_revenue.rename("Revenue").to_csv("product_revenue.csv")
region_revenue.rename("Revenue").to_csv("region_revenue.csv")
product_region.to_csv("product_region_pivot.csv")

In [42]:
# STEP 8: Print the headlines
total_revenue = df["Revenue"].sum()
best_product = product_revenue.idxmax()
best_region = region_revenue.idxmax()
best_month_start = monthly_revenue["Revenue"].idxmax()
best_month_revenue = monthly_revenue.loc[best_month_start, "Revenue"]

print("=" * 48)
print("SALES ANALYSIS HEADLINES")
print("=" * 48)
print(f"Number of sales rows: {len(df):,}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Best product: {best_product} (${product_revenue.max():,.2f})")
print(f"Best region: {best_region} (${region_revenue.max():,.2f})")
print(
    f"Best month: {best_month_start.strftime('%B %Y')} "
    f"(${best_month_revenue:,.2f})"
)
print("\\nRevenue by product:\\n", product_revenue)
print("\\nRevenue by region:\\n", region_revenue)
print("\\nProduct-by-region revenue pivot table:\\n", product_region)

SALES ANALYSIS HEADLINES
Number of sales rows: 5,000
Total revenue: $7,523,357.25
Best product: Laptop ($2,840,034.78)
Best region: East ($1,960,970.55)
Best month: December 2025 ($409,324.49)
\nRevenue by product:\n Product
Laptop        2840034.78
Phone         2678471.08
Tablet        1005075.43
Monitor        692121.93
Headphones     307654.03
Name: Revenue, dtype: float64
\nRevenue by region:\n Region
East     1960970.55
North    1881879.16
South    1876621.29
West     1803886.25
Name: Revenue, dtype: float64
\nProduct-by-region revenue pivot table:\n Region           East      North      South       West
Product                                               
Headphones   79005.20   80185.65   74285.07   74178.11
Laptop      711998.54  689486.62  762982.56  675567.06
Monitor     172965.56  182224.24  160560.09  176372.04
Phone       728806.85  666873.69  632584.02  650206.52
Tablet      268194.40  263108.96  246209.55  227562.52


In [107]:
# STEP 9: Draw the 2-by-2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Sales Analysis Dashboard 2024-2025",
             fontsize=17, fontweight="bold")

# Chart 1: Monthly revenue with 3-month rolling average
ax = axes[0, 0]
ax.plot(monthly_revenue.index, monthly_revenue["Revenue"],
        marker="o", linewidth=1.8, label="Monthly revenue")
ax.plot(monthly_revenue.index, monthly_revenue["Rolling3M"],
        linestyle="--", linewidth=2, label="3-month average")
ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.25)
ax.legend()

# Chart 2: Revenue by product
ax = axes[0, 1]
product_revenue.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Revenue by Product")
ax.set_xlabel("Revenue ($)")
ax.set_ylabel("Product")

# Chart 3: Regional revenue share
ax = axes[1, 0]
region_revenue.plot(kind="pie", autopct="%1.1f%%", startangle=90, ax=ax)
ax.set_title("Revenue Share by Region")
ax.set_ylabel("")

# Chart 4: Product-by-region heatmap
ax = axes[1, 1]
heatmap_order = ["Headphones", "Laptop", "Monitor", "Phone", "Tablet"]
region_order = ["North", "South", "East", "West"]
heatmap_data = product_region.reindex(
    index=heatmap_order, columns=region_order
)
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="Blues")
ax.set_title("Product x Region Revenue")
ax.set_xlabel("Region")
ax.set_ylabel("Product")
ax.set_xticks(range(len(region_order)), region_order)
ax.set_yticks(range(len(heatmap_order)), heatmap_order)
fig.colorbar(im, ax=ax, label="Revenue ($)");

In [109]:
# STEP 10: Tidy layout and save the dashboard PNG
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("sales_dashboard.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("\\nFiles created successfully:")
print("- sales.csv")
print("- monthly_revenue.csv")
print("- product_revenue.csv")
print("- region_revenue.csv")
print("- product_region_pivot.csv")
print("- sales_dashboard.png")

\nFiles created successfully:
- sales.csv
- monthly_revenue.csv
- product_revenue.csv
- region_revenue.csv
- product_region_pivot.csv
- sales_dashboard.png
